In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === 1. IMPORTS & DEPENDENCIES ===

# Install required packages
!pip install -q pandas numpy torch faiss-cpu sentence-transformers rank_bm25

# Core libraries
import json
import math
import re
import os
import time
import gc
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional

# FAISS for dense indexing
import faiss

# Torch and Transformers
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

# Display options
pd.set_option("display.max_colwidth", 200)

print("All dependencies installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 51.8 MB/s eta 0:00:00
All dependencies installed and imported.


In [3]:
# === 2. CONFIGURATION ===
# All file paths needed for the retriever and the benchmark

# --- 2a. Corpus & QUR Paths (Input) ---
REV5_CATALOG_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl"
REV4_CATALOG_PATH = "/content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl"
QUR_CSV_REV5 = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev5.csv"
QUR_CSV_REV4 = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_rev4.csv"
QUR_CSV_ERRB = "/content/drive/MyDrive/compliance_outputs/outputs_qur/qur_rewrites_error_bank.csv"

# --- 2b. Gold Set (Input for Sample Query) ---
REV5_GOLD_CSV = "/content/drive/MyDrive/compliance_data/gold_standard_datasets/nist800_53/nist_sp800-53_rev5_gold-set_100q.csv"

# --- 2c. Benchmark Output Path (Output) ---
BENCHMARK_OUTPUT_DIR = "/content/drive/MyDrive/ablation_outputs/performance_benchmark"
os.makedirs(BENCHMARK_OUTPUT_DIR, exist_ok=True)
print(f"Benchmark reports will be saved to: {BENCHMARK_OUTPUT_DIR}")

# --- 2d. Model & Search Parameters ---
BI_ENCODER_ID = 'intfloat/e5-small-v2'
CROSS_ENCODER_ID = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
QUR_MODEL_NAME = "Qwen2.5-7B" # User-provided model size
CANDIDATE_SET_SIZE = 50
RRF_K = 60
LATENCY_TRIALS = 10 # Number of times to run each query to get a stable average

Benchmark reports will be saved to: /content/drive/MyDrive/ablation_outputs/performance_benchmark


In [4]:
# === 3. CORE LOGIC: TEXT PROCESSING & CORPUS LOADING ===
# These functions are copied from the previous notebook to make this file self-contained.

PUNCT_RE = re.compile(r"[^0-9A-Za-z_\s]+")
WS_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    s = text.lower().replace('-', ' ')
    s = PUNCT_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

def tokenize(text: str) -> List[str]:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)
    return [t.lower() for t in TOKEN_RE.findall(text)]

class BM25Okapi:
    def __init__(self, corpus_tokens: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_tokens = corpus_tokens
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0
        self.tf: List[Dict[str, int]] = []
        df: Dict[str, int] = {}
        for doc in corpus_tokens:
            tf_doc: Dict[str, int] = {}
            for tok in doc:
                tf_doc[tok] = tf_doc.get(tok, 0) + 1
            self.tf.append(tf_doc)
            for term in tf_doc.keys():
                df[term] = df.get(term, 0) + 1
        self.idf = {term: math.log((self.N - freq + 0.5) / (freq + 0.5) + 1.0) for term, freq in df.items()}

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float32)
        for term in query_tokens:
            idf_t = self.idf.get(term)
            if idf_t is None: continue
            for i, tf_doc in enumerate(self.tf):
                f = tf_doc.get(term, 0)
                if f == 0: continue
                denom = f + self.k1 * (1 - self.b + self.b * (self.doc_len[i] / (self.avgdl + 1e-9)))
                scores[i] += idf_t * (f * (self.k1 + 1)) / (denom + 1e-9)
        return scores

    def get_top_n(self, query_tokens: List[str], n: int = 10) -> List[int]:
        scores = self.get_scores(query_tokens)
        if n >= len(scores):
            return list(np.argsort(-scores))
        idx = np.argpartition(-scores, n-1)[:n]
        idx = idx[np.argsort(-scores[idx])]
        return list(idx)

def extract_text_from_record(rec: Dict) -> str:
    texts: List[str] = []
    def recurse(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "prose" and isinstance(v, str): texts.append(v)
                else: recurse(v)
        elif isinstance(obj, list):
            for it in obj: recurse(it)
    if isinstance(rec.get("title"), str): texts.append(rec["title"])
    if isinstance(rec.get("prose"), str): texts.append(rec["prose"])
    recurse(rec.get("parts", []))
    for prm in rec.get("params") or []:
        if isinstance(prm, dict): recurse(prm.get("guidelines", []))
    return "\n".join(texts)

def extract_control_id(rec: Dict) -> Optional[str]:
    candidates = []
    for prop in rec.get("props", []) or []:
        if isinstance(prop, dict) and prop.get("name") == "label":
            val = prop.get("value")
            if isinstance(val, str) and val.strip(): candidates.append(val.strip())
    if candidates:
        def rank(v: str):
            has_paren = '(' in v
            zero_padded = bool(re.search(r"-0\d", v))
            return (has_paren, not zero_padded, len(v))
        candidates.sort(key=rank, reverse=True)
        return candidates[0]
    rid = rec.get("id")
    return rid.upper() if isinstance(rid, str) else None

def control_id_aliases(cid: str) -> List[str]:
    cid_clean = cid.strip()
    return [cid_clean, cid_clean.replace('-', ' '), cid_clean.replace('-', '')]

def load_corpus_jsonl(path: str) -> List[Dict[str, str]]:
    docs = []
    print(f"  Loading corpus from: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            rec = json.loads(line)
            cid = extract_control_id(rec)
            if not cid: continue
            body = extract_text_from_record(rec)
            if not isinstance(body, str) or not body.strip(): continue
            prefix = " ".join(control_id_aliases(cid))
            full_text = prefix + "\n" + body
            docs.append({"control_id": cid, "text": full_text})
    print(f"  Loaded {len(docs)} documents.")
    return docs

print("Core helper functions and BM25 class defined.")

Core helper functions and BM25 class defined.


In [5]:
# === 4. VRAM (RESOURCE) SNAPSHOT ===
# This cell measures the static VRAM cost of the models.

def measure_vram(model_id: str, model_class, device='cuda') -> float:
    """Loads a model, measures its VRAM, and removes it."""
    print(f"--- Measuring VRAM for {model_id} ---")

    # 1. Clean up
    gc.collect()
    torch.cuda.empty_cache()

    # 2. Load model
    print("  Loading model onto GPU...")
    model = model_class(model_id, device=device)

    # 3. Synchronize and measure
    torch.cuda.synchronize()
    mem_allocated_bytes = torch.cuda.memory_allocated()
    mem_allocated_mb = mem_allocated_bytes / 1024**2
    print(f"  VRAM Allocated: {mem_allocated_mb:.2f} MB")

    # 4. Clean up
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return mem_allocated_mb

# --- Run Measurements ---
vram_results = []
if torch.cuda.is_available():
    # Measure Bi-Encoder (S2)
    bi_encoder_vram = measure_vram(BI_ENCODER_ID, SentenceTransformer)
    vram_results.append({
        "Component": "Bi-Encoder (S2)",
        "Model": BI_ENCODER_ID,
        "VRAM (MB)": bi_encoder_vram
    })

    # Measure Cross-Encoder (S6/S7)
    cross_encoder_vram = measure_vram(CROSS_ENCODER_ID, CrossEncoder)
    vram_results.append({
        "Component": "Cross-Encoder (S6/S7)",
        "Model": CROSS_ENCODER_ID,
        "VRAM (MB)": cross_encoder_vram
    })

    # Create the final VRAM results DataFrame
    vram_results_df = pd.DataFrame(vram_results)

    print("\n--- VRAM Snapshot (Isolated) ---")
    display(vram_results_df)

    # Calculate system totals
    s2_total = bi_encoder_vram
    s6_s7_total = bi_encoder_vram + cross_encoder_vram

    print("\n--- Estimated System VRAM ---")
    print(f"  S2 (Dense) Total: {s2_total:.2f} MB")
    print(f"  S6/S7 (Hybrid) Total: {s6_s7_total:.2f} MB")

else:
    print("CUDA not available. Skipping VRAM measurement.")
    # Create an empty DataFrame for the final report
    vram_results_df = pd.DataFrame(columns=["Component", "Model", "VRAM (MB)"])

--- Measuring VRAM for intfloat/e5-small-v2 ---
  Loading model onto GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

  VRAM Allocated: 86.65 MB

--- VRAM Snapshot (Isolated) ---


,Component,Model,VRAM (MB)
0,Bi-Encoder (S2),intfloat/e5-small-v2,127.266113
1,Cross-Encoder (S6/S7),cross-encoder/ms-marco-MiniLM-L-6-v2,86.653809



--- Estimated System VRAM ---
  S2 (Dense) Total: 127.27 MB
  S6/S7 (Hybrid) Total: 213.92 MB


5

In [6]:
# === 5. THE RetrievalBenchmarker CLASS ===
# This class is modified to include separate, time-able
# methods for S1, S2, S5, S6, and S7.

class RetrievalBenchmarker:
    def __init__(self,
                 rev5_catalog_path: str,
                 rev4_catalog_path: str,
                 qur_rev5_path: str,
                 qur_rev4_path: str,
                 qur_errb_path: str,
                 device: Optional[str] = None):

        print("--- Initializing Retrieval Benchmarker ---")

        # 1. Set device
        if device:
            self.device = device
        else:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {self.device}")

        # 2. Load Models
        print(f"Loading Bi-Encoder ({BI_ENCODER_ID})...")
        self.bi_encoder = SentenceTransformer(BI_ENCODER_ID, device=self.device)
        self.bi_encoder_dim = self.bi_encoder.get_sentence_embedding_dimension()

        print(f"Loading Cross-Encoder ({CROSS_ENCODER_ID})...")
        self.cross_encoder = CrossEncoder(CROSS_ENCODER_ID, max_length=512, device=self.device)

        # 3. Build Indices
        print("\nBuilding Rev. 5 Index...")
        self.rev5_bm25, self.rev5_dense, self.rev5_map, self.rev5_ids = self._build_index(rev5_catalog_path)
        print(f"Rev. 5 Index ready: {len(self.rev5_ids)} documents.")

        print("\nBuilding Rev. 4 Index...")
        self.rev4_bm25, self.rev4_dense, self.rev4_map, self.rev4_ids = self._build_index(rev4_catalog_path)
        print(f"Rev. 4 Index ready: {len(self.rev4_ids)} documents.")

        # 4. Load QUR Data
        print("\nLoading QUR (Query Rewrite) data...")
        self.qur_rev5_df = self._load_qur_df(qur_rev5_path)
        self.qur_rev4_df = self._load_qur_df(qur_rev4_path)

        print("\n--- Benchmarker is Ready ---")

    def _load_qur_df(self, path: str) -> pd.DataFrame:
        df = pd.read_csv(path, encoding="ISO-8859-1")
        df['original_query'] = df['original_query'].astype(str).str.strip().str.lstrip(": ")
        df['rewritten_query'] = df['rewritten_query'].astype(str).str.strip().str.lstrip(": ")
        return df

    def _build_index(self, catalog_path: str) -> Tuple:
        docs = load_corpus_jsonl(catalog_path)
        id_to_text_map = {doc['control_id']: doc['text'] for doc in docs}
        control_ids = [doc['control_id'] for doc in docs]
        doc_texts = [doc['text'] for doc in docs]

        print("  Building BM25 index...")
        tokenized_corpus = [tokenize(normalize_text(t)) for t in doc_texts]
        bm25 = BM25Okapi(tokenized_corpus)

        print(f"  Encoding {len(doc_texts)} passages for Dense index...")
        passages_to_encode = [f"passage: {t}" for t in doc_texts]
        X_embeddings = self.bi_encoder.encode(
            passages_to_encode, batch_size=128, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=True
        ).astype('float32')

        print("  Building FAISS (IndexFlatIP) index...")
        dense_index = faiss.IndexFlatIP(self.bi_encoder_dim)
        dense_index.add(X_embeddings)
        return bm25, dense_index, id_to_text_map, control_ids

    def _get_rewrites(self, original_query: str, qur_df: pd.DataFrame) -> List[str]:
        q_cleaned = original_query.strip().lstrip(": ")
        matching_rows = qur_df[qur_df['original_query'] == q_cleaned]
        return matching_rows['rewritten_query'].tolist()

    def _get_components(self, version: str):
        """Helper to get the right set of indices for a query."""
        if version == 'rev5':
            return self.rev5_bm25, self.rev5_dense, self.rev5_map, self.rev5_ids, self.qur_rev5_df
        else: # Default to rev4 for this benchmark
            return self.rev4_bm25, self.rev4_dense, self.rev4_map, self.rev4_ids, self.qur_rev4_df

    # --- S1: BM25 (Baseline) ---
    def benchmark_s1(self, query: str, version: str = 'rev5', n: int = 50):
        bm25, _, _, ids, _ = self._get_components(version)
        q_tokens = tokenize(normalize_text(query))
        bm25_idxs = bm25.get_top_n(q_tokens, n=n)
        return [ids[i] for i in bm25_idxs]

    # --- S2: Dense (Baseline) ---
    def benchmark_s2(self, query: str, version: str = 'rev5', n: int = 50):
        _, dense, _, ids, _ = self._get_components(version)
        query_to_encode = [f"query: {query}"]
        q_embedding = self.bi_encoder.encode(
            query_to_encode, convert_to_numpy=True, normalize_embeddings=True
        ).astype('float32')
        scores, idxs = dense.search(q_embedding, n)
        return [ids[i] for i in idxs[0] if i != -1]

    # --- S5: Hybrid RRF (S1+S2) ---
    def benchmark_s5(self, query: str, version: str = 'rev5', n: int = 50, k: int = 60):
        bm25_results = self.benchmark_s1(query, version, n)
        dense_results = self.benchmark_s2(query, version, n)

        rrf_scores = {}
        for rank, cid in enumerate(bm25_results, start=1):
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k + rank))
        for rank, cid in enumerate(dense_results, start=1):
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k + rank))

        fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
        return [cid for cid, score in fused_ranking[:n]]

    # --- S6: Hybrid + Rerank (S5 + Reranker) ---
    def benchmark_s6(self, query: str, version: str = 'rev5', n: int = 10, k: int = 60):
        _, _, text_map, _, _ = self._get_components(version)

        # Stage 1: S5 RRF
        candidate_cids = self.benchmark_s5(query, version, n=CANDIDATE_SET_SIZE, k=k)

        # Stage 2: Rerank
        rerank_pairs = []
        rerank_cids_ordered = []
        for cid in candidate_cids:
            passage_text = text_map.get(cid)
            if passage_text:
                rerank_pairs.append((query, passage_text))
                rerank_cids_ordered.append(cid)

        if not rerank_pairs: return []

        rerank_scores = self.cross_encoder.predict(rerank_pairs, show_progress_bar=False)
        reranked_results = list(zip(rerank_cids_ordered, rerank_scores))
        reranked_results.sort(key=lambda item: item[1], reverse=True)

        return [cid for cid, score in reranked_results[:n]]

    # --- S7: ComplianceGPT (S6 + QUR) ---
    def benchmark_s7(self, query: str, version: str = 'rev5', n: int = 10, k: int = 60):
        bm25, dense, text_map, ids, qur_df = self._get_components(version)

        # Stage 1: "Superhybrid" RRF (S4 + S2)
        rrf_scores = {}

        # Part 1A: S4 (BM25 + QUR)
        all_rewrites = self._get_rewrites(query, qur_df)
        queries_to_run_bm25 = list(dict.fromkeys([query] + all_rewrites))
        for q in queries_to_run_bm25:
            q_tokens = tokenize(normalize_text(q))
            bm25_idxs = bm25.get_top_n(q_tokens, n=CANDIDATE_SET_SIZE)
            bm25_results = [ids[i] for i in bm25_idxs]
            for rank, cid in enumerate(bm25_results, start=1):
                rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k + rank))

        # Part 1B: S2 (Dense)
        dense_results = self.benchmark_s2(query, version, n=CANDIDATE_SET_SIZE)
        for rank, cid in enumerate(dense_results, start=1):
            rrf_scores[cid] = rrf_scores.get(cid, 0.0) + (1.0 / (k + rank))

        if not rrf_scores: return []
        fused_ranking = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
        candidate_cids = [cid for cid, score in fused_ranking[:CANDIDATE_SET_SIZE]]

        # Stage 2: Rerank (Same as S6)
        rerank_pairs = []
        rerank_cids_ordered = []
        for cid in candidate_cids:
            passage_text = text_map.get(cid)
            if passage_text:
                rerank_pairs.append((query, passage_text))
                rerank_cids_ordered.append(cid)

        if not rerank_pairs: return []

        rerank_scores = self.cross_encoder.predict(rerank_pairs, show_progress_bar=False)
        reranked_results = list(zip(rerank_cids_ordered, rerank_scores))
        reranked_results.sort(key=lambda item: item[1], reverse=True)

        return [cid for cid, score in reranked_results[:n]]

print("RetrievalBenchmarker class defined.")

RetrievalBenchmarker class defined.


6. Latency (Performance) Benchmark

In [7]:
# === 6. LATENCY (PERFORMANCE) BENCHMARK ===

# This cell initializes the benchmarker and times the 5 key systems.
# We run multiple trials to get a stable average, excluding the first "warmup" run.

try:
    print("--- Initializing Benchmarker... (This may take a few minutes) ---")
    benchmarker = RetrievalBenchmarker(
        REV5_CATALOG_PATH, REV4_CATALOG_PATH,
        QUR_CSV_REV5, QUR_CSV_REV4, QUR_CSV_ERRB
    )

    print("\n--- Loading Sample Query ---")
    rev5_gold = pd.read_csv(REV5_GOLD_CSV, encoding="ISO-8859-1").dropna(subset=['question'])
    # Use a standard, representative query for all tests
    sample_query = rev5_gold.iloc[5]['question']
    print(f"Sample Query: '{sample_query}'")

    # Define systems to test
    systems_to_test = {
        "S1 (BM25)": lambda: benchmarker.benchmark_s1(sample_query, 'rev5'),
        "S2 (Dense)": lambda: benchmarker.benchmark_s2(sample_query, 'rev5'),
        "S5 (Hybrid RRF)": lambda: benchmarker.benchmark_s5(sample_query, 'rev5'),
        "S6 (Hybrid + Rerank)": lambda: benchmarker.benchmark_s6(sample_query, 'rev5'),
        "S7 (ComplianceGPT)": lambda: benchmarker.benchmark_s7(sample_query, 'rev5'),
    }

    latency_results = []

    print(f"\n--- Running Latency Benchmark ({LATENCY_TRIALS} trials each) ---")

    for system_name, system_func in systems_to_test.items():
        print(f"Benchmarking: {system_name}...")
        times = []

        # Warmup run
        _ = system_func()

        for i in range(LATENCY_TRIALS):
            start_time = time.time()

            # Run the retrieval function
            _ = system_func()

            # Synchronize for GPU-based systems
            if "S2" in system_name or "S5" in system_name or "S6" in system_name or "S7" in system_name:
                torch.cuda.synchronize()

            end_time = time.time()
            times.append((end_time - start_time) * 1000) # Convert to ms

        avg_latency = np.mean(times)
        latency_results.append({
            "System": system_name,
            "Avg. Latency (ms)": avg_latency
        })

    # Create the final Latency results DataFrame
    latency_results_df = pd.DataFrame(latency_results)

    print("\n--- Latency Benchmark Results ---")
    display(latency_results_df.style.format({'Avg. Latency (ms)': '{:.2f}'}))

except Exception as e:
    print(f"\n--- CRITICAL ERROR ---")
    print(f"An unexpected error occurred: {e}")
    print("Please check all file paths in the 'Configuration' cell (Cell 2).")
    # Create an empty DataFrame for the final report
    latency_results_df = pd.DataFrame(columns=["System", "Avg. Latency (ms)"])

--- Initializing Benchmarker... (This may take a few minutes) ---
--- Initializing Retrieval Benchmarker ---
Using device: cuda
Loading Bi-Encoder (intfloat/e5-small-v2)...
Loading Cross-Encoder (cross-encoder/ms-marco-MiniLM-L-6-v2)...

Building Rev. 5 Index...
  Loading corpus from: /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl
  Loaded 1193 documents.
  Building BM25 index...
  Encoding 1193 passages for Dense index...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

  Building FAISS (IndexFlatIP) index...
Rev. 5 Index ready: 1193 documents.

Building Rev. 4 Index...
  Loading corpus from: /content/drive/MyDrive/compliance_data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl
  Loaded 922 documents.
  Building BM25 index...
  Encoding 922 passages for Dense index...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  Building FAISS (IndexFlatIP) index...
Rev. 4 Index ready: 922 documents.

Loading QUR (Query Rewrite) data...

--- Benchmarker is Ready ---

--- Loading Sample Query ---
Sample Query: 'What are the requirements for security and privacy literacy training, including its frequency, triggers for updates, and content sources?'

--- Running Latency Benchmark (10 trials each) ---
Benchmarking: S1 (BM25)...
Benchmarking: S2 (Dense)...
Benchmarking: S5 (Hybrid RRF)...
Benchmarking: S6 (Hybrid + Rerank)...
Benchmarking: S7 (ComplianceGPT)...

--- Latency Benchmark Results ---


,System,Avg. Latency (ms)
0,S1 (BM25),12.38
1,S2 (Dense),9.81
2,S5 (Hybrid RRF),23.55
3,S6 (Hybrid + Rerank),306.42
4,S7 (ComplianceGPT),367.55


7. Final Report Generation

In [8]:
# === 7. FINAL REPORT GENERATION ===
# This cell combines the VRAM and Latency results with the
# QUR model size and saves a final markdown report.

try:
    print("--- Generating Final Performance Benchmark Report ---")

    md_content = []
    md_content.append("# Retriever Performance Benchmark Report")
    md_content.append(f"Report generated: {pd.Timestamp.now()}")

    # --- 1. Latency Snapshot ---
    md_content.append("\n\n## 1. End-to-End Query Latency (Performance)")
    md_content.append(f"Average latency per query over {LATENCY_TRIALS} trials on a single GPU.")
    md_content.append("\n")
    md_content.append(latency_results_df.to_markdown(index=False, floatfmt=".2f"))

    # --- 2. VRAM Snapshot ---
    md_content.append("\n\n## 2. Model VRAM Footprint (Resource Cost)")
    md_content.append("VRAM measured by loading models onto an idle GPU.")
    md_content.append("\n")
    md_content.append(vram_results_df.to_markdown(index=False, floatfmt=".2f"))

    # Add the S2 and S6/S7 totals
    if not vram_results_df.empty:
        s2_vram = vram_results_df[vram_results_df['Component'] == 'Bi-Encoder (S2)']["VRAM (MB)"].sum()
        s6_s7_vram = vram_results_df["VRAM (MB)"].sum()

        md_content.append("\n\n### Estimated Total System VRAM")
        md_content.append(f"* **S2 (Dense) Total:** {s2_vram:.2f} MB")
        md_content.append(f"* **S6/S7 (Hybrid) Total:** {s6_s7_vram:.2f} MB")

    # --- 3. QUR Model Size ---
    md_content.append("\n\n## 3. QUR Model Size (Static Cost)")
    md_content.append("Size of the model used for Query-Rewrite (QUR) generation.")
    md_content.append("\n")
    md_content.append(f"* **QUR Model:** {QUR_MODEL_NAME}")

    # --- Save the report ---
    report_path = os.path.join(BENCHMARK_OUTPUT_DIR, "Performance_Benchmark_Report.md")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_content))

    print(f"\n\nSuccessfully saved performance report to:\n{report_path}")

except NameError as e:
    print(f"\n\n--- CRITICAL ERROR ---")
    print(f"NameError: {e}")
    print("This likely means the VRAM or Latency cells (Cell 4, Cell 6) were not run.")
    print("Please run the full notebook from the top and try again.")
except Exception as e:
    print(f"\n\nAn unexpected error occurred while saving the report: {e}")

--- Generating Final Performance Benchmark Report ---


Successfully saved performance report to:
/content/drive/MyDrive/ablation_outputs/performance_benchmark/Performance_Benchmark_Report.md
